# Phase 4 of Logistics Optimization — Basic Vehicle Routing Problem (VRP)

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

# Notebook is inside the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase4_optimization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start


In [2]:
deliveries = pd.read_csv(
    DATA_RAW / "deliveries.csv"
)

riders = pd.read_csv(
    DATA_RAW / "riders.csv"
)

distance_matrix = pd.read_csv(
    DATA_PROCESSED / "network_distance_matrix.csv",
    index_col=0
)

available_riders = riders[
    riders["available"] == True
].copy()

print("Deliveries:", len(deliveries))
print("Available riders:", len(available_riders))
print("Distance matrix shape:", distance_matrix.shape)

Deliveries: 60
Available riders: 5
Distance matrix shape: (61, 61)


In [3]:
location_ids = distance_matrix.index.tolist()

expected_locations = (
    {"DEP001"} |
    set(deliveries["delivery_id"])
)

assert distance_matrix.shape[0] == distance_matrix.shape[1], (
    "Distance matrix must be square."
)

assert distance_matrix.index.is_unique, (
    "Duplicate location IDs found in matrix rows."
)

assert distance_matrix.columns.is_unique, (
    "Duplicate location IDs found in matrix columns."
)

assert set(distance_matrix.index) == expected_locations, (
    "Matrix rows do not match depot and deliveries."
)

assert set(distance_matrix.columns) == expected_locations, (
    "Matrix columns do not match depot and deliveries."
)

# Align columns with row order
distance_matrix = distance_matrix.loc[
    location_ids,
    location_ids
]

assert distance_matrix.notna().all().all(), (
    "Distance matrix contains missing values."
)

assert np.isfinite(distance_matrix.to_numpy()).all(), (
    "Distance matrix contains infinite distances."
)

assert (distance_matrix.to_numpy() >= 0).all(), (
    "Distance matrix contains negative distances."
)

assert np.allclose(
    np.diag(distance_matrix.to_numpy()),
    0
), "Self-distances must be zero."

print("Distance matrix validation passed.")

Distance matrix validation passed.


In [4]:

print("Number of deliveries:", len(deliveries))
print("Available riders:", len(available_riders))

print(
    "Total delivery weight:",
    deliveries["weight_kg"].sum(),
    "kg"
)

print(
    "Combined rider weight capacity:",
    available_riders["capacity_kg"].sum(),
    "kg"
)

print(
    "Combined package capacity:",
    available_riders["max_packages"].sum()
)

display(
    available_riders[
        ["rider_id", "capacity_kg", "max_packages"]
    ]
)


Number of deliveries: 60
Available riders: 5
Total delivery weight: 478.3 kg
Combined rider weight capacity: 460 kg
Combined package capacity: 69


,rider_id,capacity_kg,max_packages
0,R001,100,15
1,R002,100,15
2,R003,80,12
3,R004,100,15
4,R005,80,12


In [5]:

location_ids = distance_matrix.index.tolist()

location_to_index = {
    location_id: index
    for index, location_id in enumerate(location_ids)
}

distance_matrix_int = (
    distance_matrix.round().astype(int)
)

rider_ids = available_riders["rider_id"].tolist()

num_vehicles = len(rider_ids)

depot_index = location_to_index["DEP001"]

print("Locations:", len(location_ids))
print("Vehicles:", num_vehicles)
print("Depot index:", depot_index)


Locations: 61
Vehicles: 5
Depot index: 0


In [6]:
location_to_index = {
    location_id: index
    for index, location_id in enumerate(location_ids)
}

In [7]:

manager = pywrapcp.RoutingIndexManager(
    len(location_ids),
    num_vehicles,
    depot_index
)

routing = pywrapcp.RoutingModel(manager)


In [8]:

def distance_callback(from_index, to_index):

    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)

    from_location = location_ids[from_node]
    to_location = location_ids[to_node]

    return int(
        distance_matrix_int.loc[
            from_location,
            to_location
        ]
    )


distance_callback_index = (
    routing.RegisterTransitCallback(distance_callback)
)

routing.SetArcCostEvaluatorOfAllVehicles(
    distance_callback_index
)


In [9]:

# Use grams to preserve fractional kilogram weights.
WEIGHT_SCALE = 1000

delivery_weights = (
    deliveries.set_index("delivery_id")["weight_kg"]
    .to_dict()
)

weight_demands = {
    "DEP001": 0
}

for delivery_id, weight_kg in delivery_weights.items():
    weight_demands[delivery_id] = int(
        round(float(weight_kg) * WEIGHT_SCALE)
    )

vehicle_weight_capacities = [
    int(round(float(capacity) * WEIGHT_SCALE))
    for capacity in available_riders["capacity_kg"]
]

print("Weight demand example:", list(weight_demands.items())[:5])
print("Vehicle capacities (grams):", vehicle_weight_capacities)


Weight demand example: [('DEP001', 0), ('D001', 2100), ('D002', 5200), ('D003', 9900), ('D004', 4200)]
Vehicle capacities (grams): [100000, 100000, 80000, 100000, 80000]


In [10]:

def weight_callback(from_index):

    node_index = manager.IndexToNode(from_index)

    location_id = location_ids[node_index]

    return weight_demands[location_id]


weight_callback_index = (
    routing.RegisterUnaryTransitCallback(weight_callback)
)

routing.AddDimensionWithVehicleCapacity(
    weight_callback_index,
    0,
    vehicle_weight_capacities,
    True,
    "Weight"
)


True

In [11]:

package_demands = {
    location_id: (
        0 if location_id == "DEP001" else 1
    )
    for location_id in location_ids
}

vehicle_package_capacities = (
    available_riders["max_packages"]
    .astype(int)
    .tolist()
)


def package_callback(from_index):

    node_index = manager.IndexToNode(from_index)

    location_id = location_ids[node_index]

    return package_demands[location_id]


package_callback_index = (
    routing.RegisterUnaryTransitCallback(package_callback)
)

routing.AddDimensionWithVehicleCapacity(
    package_callback_index,
    0,
    vehicle_package_capacities,
    True,
    "Packages"
)


True

In [12]:

search_parameters = (
    pywrapcp.DefaultRoutingSearchParameters()
)

search_parameters.first_solution_strategy = (
    routing_enums_pb2
    .FirstSolutionStrategy
    .PATH_CHEAPEST_ARC
)

search_parameters.local_search_metaheuristic = (
    routing_enums_pb2
    .LocalSearchMetaheuristic
    .GUIDED_LOCAL_SEARCH
)

search_parameters.time_limit.seconds = 30

solution = routing.SolveWithParameters(
    search_parameters
)

if solution is None:
    raise RuntimeError(
        "No feasible capacity-constrained solution found. "
        "Check fleet capacity and delivery demands."
    )

print("Phase 4B solution found.")


RuntimeError: No feasible capacity-constrained solution found. Check fleet capacity and delivery demands.

In [ ]:

phase4b_routes = []

for vehicle_id in range(num_vehicles):

    index = routing.Start(vehicle_id)

    route = []
    route_distance_m = 0

    while not routing.IsEnd(index):

        node = manager.IndexToNode(index)
        location_id = location_ids[node]

        route.append(location_id)

        next_index = solution.Value(
            routing.NextVar(index)
        )

        next_node = manager.IndexToNode(next_index)
        next_location = location_ids[next_node]

        route_distance_m += int(
            distance_matrix_int.loc[
                location_id,
                next_location
            ]
        )

        index = next_index

    route.append("DEP001")

    assigned_deliveries = [
        loc for loc in route
        if loc != "DEP001"
    ]

    total_weight_kg = sum(
        delivery_weights[delivery_id]
        for delivery_id in assigned_deliveries
    )

    phase4b_routes.append({
        "rider_id": rider_ids[vehicle_id],
        "route": route,
        "delivery_count": len(assigned_deliveries),
        "total_weight_kg": total_weight_kg,
        "total_distance_km": route_distance_m / 1000
    })


phase4b_routes_df = pd.DataFrame(phase4b_routes)

display(phase4b_routes_df)


In [13]:

# 1. Total delivery demand
total_weight = deliveries["weight_kg"].sum()
total_packages = len(deliveries)

# 2. Total available fleet capacity
fleet_weight_capacity = available_riders["capacity_kg"].sum()
fleet_package_capacity = available_riders["max_packages"].sum()

print("=== DELIVERY DEMAND ===")
print(f"Total deliveries: {total_packages}")
print(f"Total delivery weight: {total_weight:.2f} kg")

print("\n=== AVAILABLE FLEET ===")
print(f"Weight capacity: {fleet_weight_capacity:.2f} kg")
print(f"Package capacity: {fleet_package_capacity}")

print("\n=== CAPACITY STATUS ===")

if total_weight > fleet_weight_capacity:
    print("INSUFFICIENT WEIGHT CAPACITY")
else:
    print("Aggregate weight capacity is sufficient.")

if total_packages > fleet_package_capacity:
    print("INSUFFICIENT PACKAGE CAPACITY")
else:
    print("Aggregate package capacity is sufficient.")

print("\n=== LARGEST DELIVERY ===")
print(f"Maximum package weight: {deliveries['weight_kg'].max():.2f} kg")


=== DELIVERY DEMAND ===
Total deliveries: 60
Total delivery weight: 478.30 kg

=== AVAILABLE FLEET ===
Weight capacity: 460.00 kg
Package capacity: 69

=== CAPACITY STATUS ===
INSUFFICIENT WEIGHT CAPACITY
Aggregate package capacity is sufficient.

=== LARGEST DELIVERY ===
Maximum package weight: 21.70 kg


In [14]:

print("=== DELIVERY WEIGHT DISTRIBUTION ===")

display(
    deliveries["weight_kg"].describe()
)

print("\n=== 10 HEAVIEST DELIVERIES ===")

display(
    deliveries[
        ["delivery_id", "weight_kg"]
    ].sort_values(
        "weight_kg",
        ascending=False
    ).head(10)
)


=== DELIVERY WEIGHT DISTRIBUTION ===


count    60.000000
mean      7.971667
std       5.562102
min       1.700000
25%       3.650000
50%       5.850000
75%      10.300000
max      21.700000
Name: weight_kg, dtype: float64


=== 10 HEAVIEST DELIVERIES ===


,delivery_id,weight_kg
53,D054,21.7
45,D046,20.8
35,D036,20.5
48,D049,19.4
5,D006,19.3
52,D053,17.8
15,D016,17.1
25,D026,16.2
58,D059,15.5
47,D048,15.0


In [15]:

expected_ids = set(deliveries["delivery_id"])

weight_ids = set(weight_demands.keys()) - {"DEP001"}

package_ids = set(package_demands.keys()) - {"DEP001"}

print("Delivery IDs match weight demands:",
      expected_ids == weight_ids)

print("Delivery IDs match package demands:",
      expected_ids == package_ids)

print("Number of available riders:",
      len(rider_ids))

print("Number of weight capacities:",
      len(vehicle_weight_capacities))

print("Number of package capacities:",
      len(vehicle_package_capacities))

print("Any negative weights:",
      (deliveries["weight_kg"] < 0).any())


Delivery IDs match weight demands: True
Delivery IDs match package demands: True
Number of available riders: 5
Number of weight capacities: 5
Number of package capacities: 5
Any negative weights: False
